<a href="https://colab.research.google.com/github/AlexandraRawrrr/python_for_ds_tasks/blob/main/Copy_of_HW_12_1_%D0%86%D0%BD%D1%82%D0%B5%D0%B3%D1%80%D0%B0%D1%86%D1%96%D1%8F_Python_%D1%82%D0%B0_SQL_%D0%B7%D0%B0%D0%BF%D0%B8%D1%82%D0%B8_%D0%B4%D0%B0%D0%BD%D0%B8%D1%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Домашнє завдання: Інтеграція Python та SQL: запити даних

Це ДЗ передбачене під виконання на локальній машині. Виконання з Google Colab буде суттєво ускладнене.

## Підготовка
1. Переконайтесь, що у вас встановлены необхідні бібліотеки:
   ```bash
   pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv
   ```

2. Створіть файл `.env` з параметрами підключення до бази даних classicmodels. Базу даних ви можете отримати через

  - docker-контейнер згідно існтрукції в [документі](https://www.notion.so/hannapylieva/Docker-1eb94835849480c9b2e7f5dc22ee4df9), також відео інструкції присутні на платформі - уроки "MySQL бази, клієнт для роботи з БД, Docker і ChatGPT для запитів" та "Як встановити Docker для роботи з базами даних без терміналу"
  - або встановивши локально цю БД - для цього перегляньте урок "Опціонально. Встановлення MySQL та  БД Сlassicmodels локально".
  
  Приклад `.env` файлу ми створювали в лекції. Ось його обовʼязкове наповнення:
    ```
    DB_HOST=your_host
    DB_PORT=3306 або 3307 - той, який Ви налаштували
    DB_USER=your_username
    DB_PASSWORD=your_password
    DB_NAME=classicmodels
    ```
  Якщо ви створили цей файл під час перегляду лекції - **новий створювати не треба**. Замініть лише назву БД, або пропишіть назву в коді створення підключення (замість отримання назви цільової БД зі змінних оточення). Але переконайтесь, що до `.env` файл лежить в тій самій папці, що і цей ноутбук.

  **УВАГА!** НЕ копіюйте скрит для **створення** `.env` файлу. В лекції він наводиться для прикладу. І давалось пояснення, що в реальних проєктах ми НІКОЛИ не пишемо доступи до бази в коді. Копіювання скрипта для створення `.env` файлу сюди в ДЗ буде вважатись грубою помилкою і ми зніматимемо бали.

3. Налаштуйте підключення через SQLAlchemy до БД за прикладом в лекції.

Рекомендую вивести (відобразити) змінну engine після створення. Вона має бути не None! Якщо None - значить у Вас не підтягнулись налаштування з .env файла.

Ви також можете налаштувати параметри підключення до БД без .env файла, просто прописавши текстом в відповідних місцях. Це - не рекомендований підхід.


In [ ]:
!pip install sqlalchemy pymysql openpyxl requests python-dotenv

In [ ]:
import datetime
import requests
import json
import os

from dotenv import load_dotenv
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text, MetaData, Table
from sqlalchemy.orm import sessionmaker


In [ ]:
%%writefile .env
DB_HOST=127.0.0.1
DB_PORT=3307
DB_USER=root
DB_PASSWORD=root
DB_NAME=classicmodels

In [ ]:
# Створюємо .env файл з параметрами підключення
env_content = """
# Database Configuration
DB_HOST=127.0.0.1
DB_PORT=3307
DB_USER=root
DB_PASSWORD=root
DB_NAME=employees

# API Keys
EXCHANGE_API_KEY=your_api_key_here

# Environment
ENV=development
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("✅ Файл .env створено")
print("🔐 УВАГА: Додайте .env до .gitignore!")

In [ ]:
import socket

print("Hostname:", socket.gethostname())
print("localhost:", socket.gethostbyname_ex("localhost"))

try:
    s = socket.create_connection(("127.0.0.1", 3307), timeout=5)
    print("✅ Python видит порт 3307!")
    s.close()
except Exception as e:
    print("❌ Python НЕ видит порт 3307:", e)

In [ ]:
def create_connection():
    """
    Створює підключення через SQLAlchemy
    """
    # Завантажуємо змінні середовища
    load_dotenv()

    # Отримуємо параметри з environment variables
    host = os.getenv('DB_HOST', 'localhost')
    port = os.getenv('DB_PORT', '3306')
    user = os.getenv('DB_USER')
    password = os.getenv('DB_PASSWORD')
    database = os.getenv('DB_NAME')

    if not all([user, password, database]):
        raise ValueError("Не всі параметри БД задані в .env файлі!")

    # Створюємо connection string
    connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"

    # Створюємо engine з connection pooling
    engine = create_engine(
        connection_string,
        pool_size=2,           # Розмір пулу підключень
        max_overflow=20,        # Максимальна кількість додаткових підключень
        pool_pre_ping=True,     # Перевірка підключення перед використанням
        echo=False              # Логування SQL запитів (True для debug)
    )

    # Тестуємо підключення
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT 1"))
            result.fetchone()

        print("✅ Підключення до БД успішне!")
        print(f"🔗 {user}@{host}:{port}/{database}")
        print(f"⚡ Engine: {engine}")

        return engine

    except Exception as e:
        print(f"❌ Помилка підключення: {e}")
        return None

# Створюємо підключення
engine = create_connection()

### Завдання 1: Простий запит (1 бал)

Ми працюємо з БД Classicmodels.

**Виведіть інформацію про продукти на складі** з наступними полями:
- назва продукту (productName)
- лінійка продукту (productLine)
- кількість на складі (quantityInStock)
- ціна закупки (buyPrice)

Зчитайте дані з БД з допомогою `pd.read_sql()` з SQLAlchemy engine, який ви створили на етапі підготовки.

Виведіть перші 10 продуктів, відсортованих за кількістю на складі (від більшої кількості до меншої).

In [ ]:
query = """
SELECT
    productName,
    productLine,
    quantityInStock,
    buyPrice
FROM classicmodels.products
ORDER BY quantityInStock DESC
LIMIT 10;
"""

df = pd.read_sql(query, engine)
df


### Завдання 2: Аналітика замовлень за 2004 рік (3 бали)

**Виведіть детальну інформацію про замовлення за 2004 рік** з наступними полями:
- Номер замовлення (orderNumber)
- Дата замовлення (orderDate)
- Статус замовлення (status)
- Ім'я клієнта (customerName)
- Країна клієнта (country)
- Загальна сума замовлення (сума всіх orderdetails.quantityOrdered * orderdetails.priceEach)

Використайте JOIN для об'єднання таблиць orders, customers, orderdetails. Додайте параметризацію за роком (тільки замовлення за 2004 рік).

Використайте `text()` та named parameters для формування запиту з SQLAlchemy.

Після отримання даних з БД проведіть обчислення з Python та напишіть висновки:
1. Побудуйте стовпчасту діаграму суми замовлень по країнам і напишіть, в якій країні найбільша сума замовлень за 2004 рік.
2. В країні з найбільшою кількістю замовлень знайдіть клієнта, який зробив замовлень на найбільшу суму і виведіть імʼя цього клієнта, на яку суму він зробив замовлень і який % від всіх замовлень в цій країні становить його сума замовлень за рік.

In [ ]:
from sqlalchemy import text

query = text("""
SELECT
    o.orderNumber,
    o.orderDate,
    o.status,
    c.customerName,
    c.country,
    SUM(od.quantityOrdered * od.priceEach) AS orderTotal
FROM classicmodels.orders AS o
JOIN classicmodels.customers AS c
    ON o.customerNumber = c.customerNumber
JOIN classicmodels.orderdetails AS od
    ON o.orderNumber = od.orderNumber
WHERE YEAR(o.orderDate) = :year
GROUP BY
    o.orderNumber,
    o.orderDate,
    o.status,
    c.customerName,
    c.country
ORDER BY o.orderDate;
""")

df_orders = pd.read_sql(
    query,
    engine,
    params={"year": 2004}
)

df_orders

In [ ]:
country_sales = (
    df_orders
    .groupby("country")["orderTotal"]
    .sum()
    .sort_values(ascending=False)
)

country_sales

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

country_sales.plot(kind="bar")

plt.title("Сума замовлень за країнами у 2004 році")
plt.xlabel("Країна")
plt.ylabel("Сума замовлень, $")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Найбільша сума замовлень за 2004 рік була у США — $1,52М.

In [ ]:
usa_total = country_sales["USA"]
top_customer = usa_customers.index[0]
top_customer_total = usa_customers.iloc[0]

percentage = top_customer_total / usa_total * 100

print(f"Клієнт: {top_customer}")
print(f"Сума замовлень: ${top_customer_total:,.2f}")
print(f"Загальна сума замовлень у США: ${usa_total:,.2f}")
print(f"Частка від усіх замовлень США: {percentage:.2f}%")


### Завдання 3: Аналітичний запит - Топ продуктів по продажах (6 балів)

В цьому завданні ви отримуєте 1 бал за правильний SQL запит і по 1 балу за кожне завдання з Python.

**Проведіть аналіз прибутковості продуктів** та для цього дістаньте з БД інформацію з наступними полями:
- Назва продукту
- Лінійка продукту (productLine)
- Загальний дохід з цього продукту (сума quantity * priceEach)
- Ранг продукту по доходу (тобто яке місце посідає цей продукт за доходом серед усіх продуктів в нашому магазині)
- Який відсоток від загального доходу компанії складає цей продукт
- Різниця з середнім доходом по лінійці продукту (в %)

Відсортуйте дані за спаданням значень колонки "Який відсоток від загального доходу компанії складає цей продукт".

При створенні SQL запиту вам можуть стати в нагоді:
- **CTE** для розрахунку доходу по кожному продукту
- **Віконні функції** для ранжування та порівняння з середнім

Після отримання даних з БД проведіть обчислення (де треба) з Python та напишіть висновки:
1. Який відсоток від загального доходу складає ТОП1 продукт і що це за продукт?
2. Створіть стовпчикову діаграму топ-10 продуктів по доходу. В скільки разів відрізняється сумарний дохід за ТОП1 продуктом від 10го продукту за сумою доходу?
3. Створіть кругову діаграму розподілу доходу по лініях продуктів. Який відсоток від всіх продажів становлять продажі за ТОП2 лініями сумарно?
4. Розрахуйте та виведіть за принципом Парето (80/20) - скільки продуктів дають 80% доходу. Тобто нам треба знайти кількість продуктів сумарне значення "відсотку від загального доходу компанії", яких складає 80 починаючи з продукту з найбільшим цим відсотком.
5. Зробіть ще будь-яке аналітичне дослідження, яке дасть нам більше розуміння наших даних, що ми дістали в БД в цьому завданні. Сформоване питання до даних і обчислення має бути обовʼязково. Візуалізація - опціонально.

Візуалізацію можна створювати з будь-якою бібліотекою на ваш вибір.

In [ ]:
from sqlalchemy import text

query = text("""
WITH product_revenue AS (
    SELECT
        p.productName,
        p.productLine,
        SUM(od.quantityOrdered * od.priceEach) AS total_revenue
    FROM classicmodels.products AS p
    JOIN classicmodels.orderdetails AS od
        ON p.productCode = od.productCode
    GROUP BY
        p.productName,
        p.productLine
),

product_analysis AS (
    SELECT
        productName,
        productLine,
        total_revenue,

        RANK() OVER (
            ORDER BY total_revenue DESC
        ) AS revenue_rank,

        total_revenue /
        SUM(total_revenue) OVER () * 100 AS revenue_percent,

        AVG(total_revenue) OVER (
            PARTITION BY productLine
        ) AS avg_line_revenue

    FROM product_revenue
)

SELECT
    productName,
    productLine,
    total_revenue,
    revenue_rank,
    revenue_percent,

    (total_revenue - avg_line_revenue)
    / avg_line_revenue * 100 AS difference_from_line_avg_percent

FROM product_analysis

ORDER BY revenue_percent DESC;
""")

df_products = pd.read_sql(query, engine)

df_products

In [ ]:
top1 = df_products.iloc[0]

print(f"TOP-1 продукт: {top1['productName']}")
print(f"Дохід: ${top1['total_revenue']:,.2f}")
print(f"Частка від загального доходу: {top1['revenue_percent']:.2f}%")

In [ ]:
top10 = df_products.head(10)

top1_revenue = top10.iloc[0]["total_revenue"]
top10_revenue = top10.iloc[9]["total_revenue"]

ratio = top1_revenue / top10_revenue

print(f"TOP-1 продукт: {top10.iloc[0]['productName']}")
print(f"Дохід TOP-1: ${top1_revenue:,.2f}")
print(f"TOP-10 продукт: {top10.iloc[9]['productName']}")
print(f"Дохід TOP-10: ${top10_revenue:,.2f}")
print(f"TOP-1 приносить у {ratio:.2f} рази більше доходу, ніж TOP-10.")

In [ ]:
import matplotlib.pyplot as plt

top10 = df_products.head(10)

plt.figure(figsize=(12, 6))

plt.bar(top10["productName"], top10["total_revenue"])

plt.title("TOP-10 продуктів за доходом")
plt.xlabel("Продукт")
plt.ylabel("Дохід, $")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))

plt.pie(
    line_revenue,
    labels=line_revenue.index,
    autopct="%1.1f%%"
)

plt.title("Розподіл доходу за лініями продуктів")
plt.show()

In [ ]:
df_pareto = df_products.copy()

# Сортуємо продукти від найбільшого доходу до найменшого
df_pareto = df_pareto.sort_values(
    "total_revenue",
    ascending=False
)

# Кумулятивна частка доходу
df_pareto["cumulative_percent"] = (
    df_pareto["total_revenue"].cumsum()
    / df_pareto["total_revenue"].sum()
    * 100
)

# Кількість продуктів, необхідна для досягнення 80% доходу
products_for_80 = (
    df_pareto["cumulative_percent"] < 80
).sum() + 1

total_products = len(df_pareto)

print(f"Усього продуктів: {total_products}")
print(f"Для отримання 80% доходу потрібно продуктів: {products_for_80}")
print(
    f"Це становить "
    f"{products_for_80 / total_products * 100:.2f}% "
    f"від усіх продуктів."
)

In [ ]:
line_analysis = (
    df_products
    .groupby("productLine")
    .agg(
        total_revenue=("total_revenue", "sum"),
        products_count=("productName", "count")
    )
)

line_analysis["avg_revenue_per_product"] = (
    line_analysis["total_revenue"]
    / line_analysis["products_count"]
)

line_analysis = line_analysis.sort_values(
    "avg_revenue_per_product",
    ascending=False
)

line_analysis

### ОПЦІОНАЛЬНО. Завдання 4: Аналітичний запит - Динаміка продажів по місяцях (8 балів)

**Проведіть аналіз динаміки продажів по місяцях** та для цього дістаньте з бази інформацію з наступними полями:
- Рік та місяць замовлень
- Кількість замовлень за цей рік-місяць
- Загальний дохід (quantityOrdered * priceEach) за цей рік-місяць
- На який % ми зросли за доходом порівняно з попереднім місяцем
- Накопичувальний дохід до цього місяця за рік
- Ковзне середнє доходу за 3 місяці
- Ранг цього місяця за доходом

Використайте:
- **CTE** для агрегації продажів по місяцях
- **Віконні функції** для розрахунку:
  - Зростання доходу порівняно з попереднім місяцем (LAG)
  - Накопичувальний дохід за рік
  - Ковзне середнє доходу за 3 місяці (AVG OVER)
  - Ранжування місяців за доходом (RANK)

Після отримання даних з БД побудуйте наступні графіки і напишіть коротко, які висновки з них можна зробити.
1. Створіть лінійний графік доходу по місяцях. Чи є тред до зростання в даних?
2. Створіть графік місяць-до-місяця зростання у відсотках.
3. Створіть heatmap сезонності (місяць vs рік)
4. Відобразіть козвне середнє разом з динамікою продажів.
5. Розрахуйте кореляцію між кількістю замовлень та середнім чеком та побудуйте графік розсіювання між цими змінними. Чи є лінійна залежність?

Очікуваний результат кожного графіку - нижче.
В першому графіку я додала лінію тренду аби показати тренд. Вам її додавати не треба.

**ВАЖЛИВО!** Якщо ви захочете назвати одну з колонок .`year_month` - запит не буде працювати, бо year_month - [зарезервоване слово](https://stackoverflow.com/questions/34906190/mysql-using-year-month) в SQL і сприймається як команда. Треба використати іншу назву для відповідної колонки.


In [ ]:
from sqlalchemy import text

query_monthly = text("""
WITH monthly_sales AS (
    SELECT
        YEAR(o.orderDate) AS order_year,
        MONTH(o.orderDate) AS order_month,
        SUM(od.quantityOrdered) AS orders_quantity,
        SUM(od.quantityOrdered * od.priceEach) AS total_revenue
    FROM classicmodels.orders AS o
    JOIN classicmodels.orderdetails AS od
        ON o.orderNumber = od.orderNumber
    GROUP BY
        YEAR(o.orderDate),
        MONTH(o.orderDate)
)

SELECT
    order_year,
    order_month,
    orders_quantity,
    total_revenue,

    LAG(total_revenue) OVER (
        PARTITION BY order_year
        ORDER BY order_month
    ) AS previous_month_revenue,

    (
        total_revenue -
        LAG(total_revenue) OVER (
            PARTITION BY order_year
            ORDER BY order_month
        )
    )
    /
    LAG(total_revenue) OVER (
        PARTITION BY order_year
        ORDER BY order_month
    ) * 100 AS revenue_growth_percent,

    SUM(total_revenue) OVER (
        PARTITION BY order_year
        ORDER BY order_month
    ) AS cumulative_revenue,

    AVG(total_revenue) OVER (
        PARTITION BY order_year
        ORDER BY order_month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3_month_avg,

    RANK() OVER (
        PARTITION BY order_year
        ORDER BY total_revenue DESC
    ) AS revenue_rank

FROM monthly_sales

ORDER BY order_year, order_month;
""")

df_monthly = pd.read_sql(query_monthly, engine)

df_monthly

In [ ]:
# 1. Лінійний графік доходу з трендом (аби показати, що тренд є)
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    range(len(df_monthly)),
    df_monthly["total_revenue"],
    marker="o"
)

labels = (
    df_monthly["order_year"].astype(str)
    + "-"
    + df_monthly["order_month"].astype(str).str.zfill(2)
)

plt.xticks(range(len(df_monthly)), labels, rotation=45)

plt.title("Динаміка доходу за місяцями")
plt.xlabel("Місяць")
plt.ylabel("Дохід, $")

plt.tight_layout()
plt.show()

In [ ]:
# 2. Місяць-до-місяця зростання
plt.figure(figsize=(12, 6))

plt.bar(
    range(len(df_monthly)),
    df_monthly["revenue_growth_percent"],
    color="purple"
)

labels = (
    df_monthly["order_year"].astype(str)
    + "-"
    + df_monthly["order_month"].astype(str).str.zfill(2)
)

plt.xticks(range(len(df_monthly)), labels, rotation=45)

plt.title("Місяць-до-місяця зростання доходу")
plt.xlabel("Місяць")
plt.ylabel("Зростання доходу, %")

plt.axhline(0, linewidth=1)

plt.tight_layout()
plt.show()

In [ ]:
# 3. Heatmap сезонності
import seaborn as sns
import matplotlib.pyplot as plt

seasonality = df_monthly.pivot(
    index="order_month",
    columns="order_year",
    values="total_revenue"
)

plt.figure(figsize=(12, 7))

sns.heatmap(
    seasonality,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu"
)

plt.title("Сезонність продажів за місяцями та роками")
plt.xlabel("Рік")
plt.ylabel("Місяць")

plt.tight_layout()
plt.show()

In [ ]:
# 4. Ковзне середнє з динімікою доходу
plt.figure(figsize=(12, 6))

plt.plot(
    range(len(df_monthly)),
    df_monthly["total_revenue"],
    marker="o",
    label="Фактичний дохід"
)

plt.plot(
    range(len(df_monthly)),
    df_monthly["rolling_3_month_avg"],
    marker="o",
    color="orange",
    linewidth=3,
    label="Ковзне середнє за 3 місяці"
)

labels = (
    df_monthly["order_year"].astype(str)
    + "-"
    + df_monthly["order_month"].astype(str).str.zfill(2)
)

plt.xticks(range(len(df_monthly)), labels, rotation=45)

plt.title("Динаміка доходу та ковзне середнє за 3 місяці")
plt.xlabel("Місяць")
plt.ylabel("Дохід, $")

plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 5. Кореляція замовлень та середнього чеку
monthly_ranking = df_monthly.copy()

monthly_ranking["month_name"] = (
    monthly_ranking["order_year"].astype(str)
    + "-"
    + monthly_ranking["order_month"].astype(str).str.zfill(2)
)

monthly_ranking = monthly_ranking.sort_values(
    "total_revenue",
    ascending=False
)

plt.figure(figsize=(12, 6))

plt.bar(
    monthly_ranking["month_name"].head(10),
    monthly_ranking["total_revenue"].head(10),
    color="turquoise"
)

plt.title("TOP-10 місяців за доходом")
plt.xlabel("Місяць")
plt.ylabel("Дохід, $")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
monthly_ranking[
    ["month_name", "total_revenue", "revenue_rank"]
].head(10)

Є аутлаєри, але в загальному не можна сказати, що кореляція присутня.